In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import Tool
from langchain.agents import create_agent
from langchain_tavily import TavilySearch

# 🔐 Load API keys
load_dotenv(".env")

# 🔸 Initialize LLM
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

#  Tool 1 : Simple QA

qa_prompt = ChatPromptTemplate.from_template(
    "Answer clearly: {question}"
)

qa_chain = qa_prompt | llm

qa_tool = Tool(
    name="Simple_QA",
    func=lambda question: qa_chain.invoke(
        {"question": question}
    ).content,
    description="Answers factual questions clearly."
)

#  Tool 2 : Summarizer

summary_prompt = ChatPromptTemplate.from_template(
    "Summarize this text:\n\n{text}"
)

summary_chain = summary_prompt | llm

summary_tool = Tool(
    name="Summarizer",
    func=lambda text: summary_chain.invoke(
        {"text": text}
    ).content,
    description="Summarizes long paragraphs or text."
)

# Tool 3 : Tavily Web Search

search_tool = TavilySearch(
    max_results=3,
    tavily_api_key=os.getenv("TAVILY_API_KEY")
)

# Create Agent

tools = [
    qa_tool,
    summary_tool,
    search_tool
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful AI assistant. "
        "Use the appropriate tool whenever it improves the answer."
    )
)

# Run Queries

queries = [
    "What is LangGraph in LangChain?"]

for query in queries:
    print("\n User Query:", query)
    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": query
                }
            ]
        }
    )
    print("Action: simple_qa")
    print("Action Input:", query)
    print("\n Agent Response:\n")
    print(response["messages"][-1].content)



 User Query: What is LangGraph in LangChain?
Action: simple_qa
Action Input: What is LangGraph in LangChain?

 Agent Response:

LangGraph in LangChain is an open-source AI agent framework designed to build, deploy, and manage complex generative AI agent workflows. It extends LangChain's capabilities to support complex, stateful workflows with loops, branches, and multiple agents, modeling workflows as cyclic graphs where nodes represent actions and edges define the flow between them. This allows dynamic routing based on runtime conditions.

LangGraph uses graph-based architectures to model and manage intricate relationships between various components of an AI agent workflow. It provides tools and libraries to create, run, and optimize large language models (LLMs) in a scalable and efficient manner. LangGraph is complementary to LangChain, with LangChain focusing on simpler sequential pipelines and LangGraph enabling more sophisticated AI applications with logic, control, and flow.

Ke